# Visualization Basics

Checks the lightweight graph display entry points and keeps rendering output bounded for reviewable notebooks.

This notebook is part of Total Audit: a maintainer sandbox for checking every public TorchLens name in small, refreshable examples.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
model = tiny_model(seed=0).eval()
x = torch.randn(1, 4)
with torch.no_grad():
    log = tl.log_forward_pass(model, x)
print(type(log).__name__)
print(log.layer_labels_no_pass[:5])
inline_show("saved layers", len(log.layers_with_saved_activations))

Try changing the seed, batch size, or layer label in the next cell. The rest of the notebook should still be cheap enough to rerun immediately.

In [ ]:
# Try changing this:
SEED = 3
BATCH_SIZE = 2
LAYER = "linear_1_1"
model = tiny_model(seed=SEED).eval()
stimuli = torch.randn(BATCH_SIZE, 4)
activation = tl.peek(model, stimuli[:1], LAYER)
print(LAYER, tuple(activation.shape))

In [ ]:
cnn = tiny_cnn(seed=1).eval()
image = torch.randn(1, 1, 6, 6)
with torch.no_grad():
    cnn_log = tl.log_forward_pass(cnn, image)
print(cnn_log.layer_labels_no_pass[:4])

sequence_model = tiny_recurrent(seed=2).eval()
sequence = torch.randn(1, 3, 3)
with torch.no_grad():
    recurrent_log = tl.log_forward_pass(sequence_model, sequence)
print(recurrent_log.layer_labels_no_pass[:4])

In [ ]:
try:
    tl.peek(tiny_model(seed=0).eval(), torch.randn(1, 4), "definitely_missing_layer")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc).split(".")[0])

In [ ]:
def reset_tmpdir() -> Path:
    """Reset the notebook temporary directory."""

    global TMPDIR
    if TMPDIR.exists():
        shutil.rmtree(TMPDIR)
    TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
    return TMPDIR


print(reset_tmpdir().name)

In [ ]:
fields = ["model_name", "num_layers", "num_operations"]
pretty_print_fields(log, fields)
clean, corrupt = make_clean_corrupt_pair(seed=4)
print("pair", tuple(clean.shape), tuple(corrupt.shape), torch.allclose(clean, corrupt))

In [ ]:
COVERAGE_CALLS = [
    "tl.LayerLog.transformed_gradient_shape",
    "tl.LayerLog.uses_params",
    "tl.LayerPassLog",
    "tl.LayerPassLog.DEFAULT_FILL_STATE",
    "tl.LayerPassLog.FORK_POLICY",
    "tl.LayerPassLog.PORTABLE_STATE_SPEC",
    "tl.LayerPassLog._construction_done",
    "tl.LayerPassLog._pass_finished",
    "tl.LayerPassLog.activation",
    "tl.LayerPassLog.activation_postfunc",
    "tl.LayerPassLog.activation_transform",
    "tl.LayerPassLog.args_captured",
    "tl.LayerPassLog.autograd_saved_bytes",
    "tl.LayerPassLog.autograd_saved_tensor_count",
    "tl.LayerPassLog.bool_conditional_id",
    "tl.LayerPassLog.bool_context_kind",
    "tl.LayerPassLog.bool_is_branch",
    "tl.LayerPassLog.bool_wrapper_kind",
    "tl.LayerPassLog.buffer_address",
    "tl.LayerPassLog.buffer_parent",
    "tl.LayerPassLog.buffer_pass",
    "tl.LayerPassLog.bytes_delta_at_call",
    "tl.LayerPassLog.bytes_peak_at_call",
    "tl.LayerPassLog.captured_arg_template",
    "tl.LayerPassLog.captured_args",
    "tl.LayerPassLog.captured_kwarg_template",
    "tl.LayerPassLog.captured_kwargs",
    "tl.LayerPassLog.child_layers",
    "tl.LayerPassLog.children_tensor_versions",
    "tl.LayerPassLog.co_parent_layers",
    "tl.LayerPassLog.cond_branch_children_by_cond",
    "tl.LayerPassLog.cond_branch_elif_children",
    "tl.LayerPassLog.cond_branch_else_children",
    "tl.LayerPassLog.cond_branch_start_children",
    "tl.LayerPassLog.cond_branch_then_children",
    "tl.LayerPassLog.conditional_branch_depth",
    "tl.LayerPassLog.conditional_branch_stack",
    "tl.LayerPassLog.container_spec",
    "tl.LayerPassLog.containing_module",
    "tl.LayerPassLog.containing_modules",
    "tl.LayerPassLog.copy",
    "tl.LayerPassLog.corresponding_grad_fn",
    "tl.LayerPassLog.creation_order",
    "tl.LayerPassLog.detach_saved_tensor",
    "tl.LayerPassLog.edge_uses",
    "tl.LayerPassLog.equivalent_operations",
    "tl.LayerPassLog.extra_data",
    "tl.LayerPassLog.feeds_output",
    "tl.LayerPassLog.flops_backward",
    "tl.LayerPassLog.flops_forward",
    "tl.LayerPassLog.func_applied",
    "tl.LayerPassLog.func_argnames",
    "tl.LayerPassLog.func_autocast_state",
    "tl.LayerPassLog.func_call_id",
    "tl.LayerPassLog.func_call_stack",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")